# ATC / ADAPT — Colab: SFT (JSON) + GRPO for medium tier and negotiation

This notebook follows the same layout as `training/atc_multiagent_colab.ipynb` in this repo, and adds an **optional SFT cold-start** phase (`training/build_sft_dataset.py` + `training/train_sft.py`) before **GRPO** (`training/train_grpo.py`).

**Reference notebook (your link):** [Google Colab drive](https://colab.research.google.com/drive/11C0POK25U1hTRi-KwIV3HaYYYCWJL7tL?usp=sharing)

**Use case:** improve Qwen 2.5–7B on Tier 1–3 and negotiation (e.g. `mumbai_bank_balance_medium`, ADAPT ICU) where zero-shot JSON and coordination break down.

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (T4 works for short runs; L4/A100 recommended for 7B + 200+ GRPO episodes).


## 0 — Colab secrets (recommended)

Colab sidebar → **Secrets** → create `HF_TOKEN` for Hugging Face downloads and HF Inference Router (if you use API inference later).


In [ ]:
try:
    from google.colab import userdata
    import os
    t = userdata.get("HF_TOKEN")
    if t:
        os.environ["HF_TOKEN"] = t
        os.environ["HUGGING_FACE_HUB_TOKEN"] = t
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("No HF_TOKEN secret — add one for gated models / router.")
except Exception as e:
    print("Secrets unavailable (not on Colab?)", e)


## 1 — Paths and hyperparameters

Edit **`REPO_URL`** and **`GIT_BRANCH`** below (defaults: `yvishi/ats`, branch `cleaned`). With `USE_DRIVE=True`, checkpoints land under **Google Drive**.

**Quick smoke run (see same cell):** set `RUN_SFT = False` and lower `GRPO_EPISODES` / `EVAL_EPISODES` to sanity-check curves without long runs.


In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
# --- Git: set your clone URL and branch name here ------------------------
REPO_URL = "https://github.com/yvishi/ats.git"
GIT_BRANCH = "cleaned"   # e.g. "main", "cleaned"
REPO_DIR = Path("/content/ATS")

OUTPUT_DIR = Path("/content/drive/MyDrive/atc-grpo-medium") if USE_DRIVE else Path("/content/atc-grpo-medium")
SFT_DATA = Path("/content/drive/MyDrive/atc-sft-data.jsonl") if USE_DRIVE else Path("/content/atc-sft-data.jsonl")
SFT_ADAPTER_DIR = Path("/content/drive/MyDrive/atc-sft-json-adapter") if USE_DRIVE else Path("/content/atc-sft-json-adapter")

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
LORA_RANK = 16

# Toggle this for fast turnaround analysis runs.
QUICK_ANALYSIS = True

if QUICK_ANALYSIS:
    RUN_SFT = False
    SFT_EPISODES = 200
    SFT_EPOCHS = 0.5
    SFT_MAX_SEQ_LENGTH = 1024
    SFT_BATCH_SIZE = 1
    SFT_GRAD_ACCUM = 4

    GRPO_EPISODES = 40
    GRPO_BATCH_SIZE = 2
    GRPO_GRAD_ACCUM = 1
    GRPO_N_GENERATIONS = 4
    GRPO_MAX_NEW_TOKENS = 256
    GRPO_LOGGING_STEPS = 1

    EVAL_EPISODES = 3
    RUN_EVAL = False
else:
    RUN_SFT = True
    SFT_EPISODES = 800
    SFT_EPOCHS = 1.0
    SFT_MAX_SEQ_LENGTH = 2048
    SFT_BATCH_SIZE = 1
    SFT_GRAD_ACCUM = 8

    GRPO_EPISODES = 150
    GRPO_BATCH_SIZE = 4
    GRPO_GRAD_ACCUM = 2
    GRPO_N_GENERATIONS = 8
    GRPO_MAX_NEW_TOKENS = 384
    GRPO_LOGGING_STEPS = 1

    EVAL_EPISODES = 8
    RUN_EVAL = True

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

print("OUTPUT_DIR", OUTPUT_DIR)
print("QUICK_ANALYSIS", QUICK_ANALYSIS)
print("RUN_SFT", RUN_SFT, "SFT_EPOCHS", SFT_EPOCHS, "SFT_EPISODES", SFT_EPISODES)
print("GRPO_EPISODES", GRPO_EPISODES, "B", GRPO_BATCH_SIZE, "GA", GRPO_GRAD_ACCUM, "G", GRPO_N_GENERATIONS)
print("RUN_EVAL", RUN_EVAL, "EVAL_EPISODES", EVAL_EPISODES)


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SFT_ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
print("Ready:", OUTPUT_DIR)


## 2 — Clone repository


In [ ]:
import shutil, subprocess, os, sys
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("cwd:", Path.cwd())


## 3 — Install dependencies (Unsloth + TRL)

If you hit import/version errors after this cell, use **Runtime → Restart session** and re-run from §2.


In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)

pip("pip")
pip("unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git")
pip("trl>=0.9.6", "datasets>=2.20.0", "accelerate>=0.32.0", "peft>=0.12.0", "bitsandbytes>=0.43.0")
pip("matplotlib>=3.9.0", "numpy>=1.26.0")
pip("openenv-core[core]>=0.2.3", "fastapi>=0.128.0", "openai>=2.30.0", "pydantic>=2.12.0", "uvicorn>=0.41.0")
print("pip install complete")


## 4 — GPU check + GRPO dataset sanity


In [ ]:
import torch
from training.dataset import build_episode_dataset

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → GPU)."
print("GPU:", torch.cuda.get_device_name(0))

rows = build_episode_dataset(
    n_episodes=4,
    seed=SEED,
    include_adapt=True,
    include_generator=False,
    include_supervisor=False,
)
print("rows:", len(rows), "roles:", sorted({r["agent_role"] for r in rows}))


## 5 — Phase A (optional): SFT on strict JSON

Builds `SFT_DATA` jsonl with **teacher** heuristics, then runs `training/train_sft.py` into `SFT_ADAPTER_DIR`. This targets **parseable** AMAN/DMAN/ADAPT JSON before RL.

SFT training prints live and emits explicit `[SFT] EPOCH END` summaries during training.


In [ ]:
import subprocess, sys

if not RUN_SFT:
    print("Skipping SFT (set RUN_SFT=True to enable).")
else:
    r0 = subprocess.run([
        sys.executable, "training/build_sft_dataset.py",
        "--out", str(SFT_DATA),
        "--episodes", str(SFT_EPISODES),
        "--seed", str(SEED),
    ], cwd=str(REPO_DIR), capture_output=True, text=True)
    if r0.returncode != 0:
        print(r0.stdout)
        print(r0.stderr)
        r0.check_returncode()
    # In-process: full Python tracebacks show in the notebook (subprocess hides them).
    _old_argv = sys.argv[:]
    try:
        sys.argv = [
            "train_sft.py",
            "--dataset", str(SFT_DATA),
            "--output_dir", str(SFT_ADAPTER_DIR),
            "--model", MODEL_NAME,
            "--epochs", str(SFT_EPOCHS),
            "--max_seq_length", str(SFT_MAX_SEQ_LENGTH),
            "--batch_size", str(SFT_BATCH_SIZE),
            "--grad_accum", str(SFT_GRAD_ACCUM),
            "--lora_rank", str(LORA_RANK),
            "--seed", str(SEED),
            "--trainer", "hf",
        ]
        print("Running:", " ".join([sys.executable] + sys.argv), flush=True)
        from training.train_sft import main as _train_sft_main
        _train_sft_main()
    finally:
        sys.argv = _old_argv
    print("SFT adapter:", SFT_ADAPTER_DIR)


## 6 — Phase B: GRPO multi-agent training

Runs the full `training/train_grpo.py` loop (curriculum generator, AMAN/DMAN rewards, optional ADAPT in dataset). **Increase `GRPO_EPISODES`** on larger GPUs for better medium-tier convergence.

This cell runs GRPO **in-process** so logs stream live in Colab, including `[LIVE]` updates and explicit `[GRPO] EPOCH END` prints.

Note: GRPO currently starts a **fresh** LoRA on the base model. The SFT adapter in §5 is still valuable for **separate** inference or future `train_grpo` init-adapter support; see README “SFT cold-start”.


In [ ]:
import os, sys

# Run in-process so per-step and per-epoch logs stream directly in the notebook.
os.environ.setdefault("PYTHONUNBUFFERED", "1")
_old_argv = sys.argv[:]
try:
    sys.argv = [
        "train_grpo.py",
        "--model", MODEL_NAME,
        "--output_dir", str(OUTPUT_DIR),
        "--episodes", str(GRPO_EPISODES),
        "--lora_rank", str(LORA_RANK),
        "--seed", str(SEED),
        "--eval_episodes", str(EVAL_EPISODES),
        "--batch_size", str(GRPO_BATCH_SIZE),
        "--grad_accum", str(GRPO_GRAD_ACCUM),
        "--n_generations", str(GRPO_N_GENERATIONS),
        "--max_new_tokens", str(GRPO_MAX_NEW_TOKENS),
        "--logging_steps", str(GRPO_LOGGING_STEPS),
    ]
    if not RUN_EVAL:
        sys.argv.append("--no_eval")
    print("Running:", " ".join([sys.executable] + sys.argv), flush=True)
    from training.train_grpo import main as _train_grpo_main
    _train_grpo_main()
finally:
    sys.argv = _old_argv

print("Saved:", OUTPUT_DIR)


## 7 — Evaluation: base vs trained (`training/eval.py`)


In [ ]:
import subprocess, sys
from pathlib import Path

if not RUN_EVAL:
    print("Skipping eval cell for quick analysis (set RUN_EVAL=True to enable).")
else:
    eval_json = OUTPUT_DIR / "eval_results_colab.json"
    subprocess.run([
        sys.executable, "training/eval.py",
        "--base", MODEL_NAME,
        "--trained", str(OUTPUT_DIR),
        "--episodes", str(EVAL_EPISODES),
        "--output", str(eval_json),
    ], cwd=str(REPO_DIR), check=True)
    print(eval_json.read_text()[:2000])


## 8 — Reward curves and eval plots


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

for argv in (
    ["training/plot_rewards.py", "--input", str(OUTPUT_DIR / "reward_curves.json"), "--save", str(plots_dir), "--no_show"],
    ["training/plot_rewards.py", "--eval_results", str(OUTPUT_DIR / "eval_results.json"), "--save", str(plots_dir), "--no_show"],
):
    subprocess.run([sys.executable] + argv, cwd=str(REPO_DIR), check=False)

for name in ("training_curves.png", "eval_comparison.png"):
    p = plots_dir / name
    if p.exists():
        display(Image(filename=str(p)))


## 9 — Optional: push adapter to Hugging Face Hub

Uncomment and set `HUB_REPO_ID` after `huggingface_hub` login.


In [ ]:
# from huggingface_hub import HfApi
# import os
# HUB_REPO_ID = "your-user/atc-grpo-medium-7b"
# api = HfApi(token=os.environ.get("HF_TOKEN"))
# api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=HUB_REPO_ID, repo_type="model")
# print("Uploaded to", HUB_REPO_ID)
print("Hub upload cell is commented out by default.")


## Troubleshooting

| Issue | What to do |
|-------|------------|
| CUDA OOM on 7B | Lower `GRPO_EPISODES`; in SFT use `--batch_size 1`; try smaller model or A100 |
| `trl` / `GRPOConfig` keyword errors | Restart runtime; pin `trl` to version compatible with your Unsloth build |
| Clone permission denied | Use a **public** `REPO_URL` or embed a read token in the git URL |
| `eval.py` cannot load trained folder | Ensure `OUTPUT_DIR` contains `adapter_config.json` from GRPO save |

**Local MetaHack repo:** If your canonical code lives at `F:\College\SEM4\MetaHack\ats`, push it to GitHub and set `REPO_URL` to that remote so Colab matches your branch.
